# Colab 38 - where the protocol constants come from

**Why:** supervisor note 67 calls the 200,000 / 400-per-decile numbers *"very arbitrary"*, and the
same question applies to the 20,000 + 8,000 split. No rationale for any of them is recorded anywhere
in the repository - the only documented justification is that they are held **identical across every
arm**, which justifies keeping them, not choosing them.

This notebook measures what each constant actually buys, so Section 3.6 can state a criterion instead
of a preference.

| | question | experiment |
|---|---|---|
| A | why 8,000 independent pairs, could it be lower | sweep `n_indep`, watch the evaluation set size |
| B | why 400 per decile | sweep the cap, watch which deciles saturate |
| C | does the cap bind on the CATH data too | per-decile supply in the sampled candidates |
| D | was 20,000 + 10,000 sized to match the 30,000 training pairs | distance to the training profile |

**What it does NOT do:** no model, no training, no metrics. Counting only.

**History, for context.** colab29 used 30,000 altered + 10,000 independent and 300,000 candidates.
From colab31 onward these became 20,000 + 8,000 and 200,000, and were carried forward unchanged
through colab36. colab29's receipt records decile counts `[45, 400x9]` - nine deciles capped, the
lowest supplying 45 - *with* 10,000 independent pairs.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')
!pip install rapidfuzz --quiet


In [ ]:
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
from rapidfuzz.distance import Levenshtein as RFLev

DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'

# --- IDENTICAL to the run of record ---
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
MIN_LEN, MAX_LEN = 50, 200
RESCUED    = {'4z0mC02', '3qkaE02'}
N_TRAIN    = 30_000
TRAIN_SEED = 0
SYN_PERTURB, SYN_INDEP = 20_000, 8_000     # the constants under examination
STRAT_PER_BIN          = 400               # ditto
STRAT_CAND, PAIR_SEED  = 200_000, 999
SYN_SEED   = 20260810
DECILES    = np.linspace(0, 1, 11)

COLOUR = {'Synth': '#FF7F0E', '3Di': '#0072B2', 'SS': '#D62728', 'AA': '#4D4D4D'}

AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s)
is_ss = lambda s: all(c in SS_SET for c in s)

def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L

def decile_of(nl):
    return np.clip(np.digitize(nl, DECILES) - 1, 0, 9)

def occupancy(nl):
    return np.bincount(decile_of(np.asarray(nl)), minlength=10)

def balanced_size(counts, cap):
    return int(np.minimum(counts, cap).sum())


## 2. Reference sets

Generated once and reused by every experiment. The independent-pair pool is drawn at its maximum size
and then subsampled as a prefix, so the sweep in Experiment A is nested: differences between rows are
the effect of the count, not of a different random draw.

In [ ]:
# --- generator, verbatim from the run of record ---
def rand_seq(abc, rng):
    L = int(rng.integers(MIN_LEN, MAX_LEN + 1))
    return ''.join(rng.choice(list(abc), size=L))

def perturb(seq, k, abc, rng):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0:            op = 'ins'
        elif len(s) >= MAX_LEN:    op = rng.choice(['sub', 'del'])
        else:                      op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        else:
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def training_labels(n, seed):
    rng = np.random.default_rng(seed); out = []
    while len(out) < n:
        sd = rand_seq(AA_ALPHABET, rng); L = len(sd)
        t = float(rng.uniform(0, 1)); k = max(0, int(round((1 - t) * L)))
        o = perturb(sd, k, AA_ALPHABET, rng)
        if 1 <= len(o) <= MAX_LEN:
            out.append(norm_lev(sd, o))
    return np.array(out)

def synth_scores(n_perturb, n_indep, seed=SYN_SEED):
    """normLev of the generated pairs, split by kind. No balancing."""
    r = np.random.default_rng(seed); alt, ind = [], []
    for _ in range(n_perturb):
        base = rand_seq(AA_ALPHABET, r)
        part = perturb(base, int(r.integers(0, len(base) + 1)), AA_ALPHABET, r)
        if 1 <= len(part) <= MAX_LEN:
            alt.append(norm_lev(base, part))
    for _ in range(n_indep):
        ind.append(norm_lev(rand_seq(AA_ALPHABET, r), rand_seq(AA_ALPHABET, r)))
    return np.array(alt), np.array(ind)

print('generating the reference sets once (about two minutes)...')
TRAIN_NL = training_labels(N_TRAIN, TRAIN_SEED)
ALT, IND_MAX = synth_scores(SYN_PERTURB, 20_000)   # 20,000 independent, subsampled below
print(f'  training pairs   {len(TRAIN_NL):,}   median {np.median(TRAIN_NL):.3f}')
print(f'  altered pairs    {len(ALT):,}   median {np.median(ALT):.3f}')
print(f'  independent pool {len(IND_MAX):,}   median {np.median(IND_MAX):.3f}')
print('')
print(f'  independent-pair range: [{IND_MAX.min():.3f}, {IND_MAX.max():.3f}]')
print(f'  independent pairs below 0.10: {int((IND_MAX < 0.10).sum())} of {len(IND_MAX):,}')
print(f'  altered pairs below 0.10:     {int((ALT < 0.10).sum())} of {len(ALT):,}')


## 3. Experiment A - the independent-pair count

The independent pairs exist to populate the low similarity range. The question is how many are needed
before that range stops growing.

In [ ]:
# EXPERIMENT A - how many independent pairs are actually needed?
# n_perturb fixed at 20,000; n_indep swept. The independent pairs are drawn as a
# prefix of one pool, so the sweep is nested and differences are not sampling noise.
rows = []
for n_ind in [0, 1_000, 2_000, 4_000, 6_000, 8_000, 10_000, 15_000, 20_000]:
    nl = np.concatenate([ALT, IND_MAX[:n_ind]])
    occ = occupancy(nl)
    rows.append(dict(n_indep=n_ind, generated=len(nl),
                     **{f'd{i}': int(occ[i]) for i in range(10)},
                     deciles_at_cap=int((occ >= STRAT_PER_BIN).sum()),
                     balanced=balanced_size(occ, STRAT_PER_BIN)))
SWEEP_A = pd.DataFrame(rows)
print('Decile occupancy before capping (d0 = [0.0,0.1), d9 = [0.9,1.0]):')
print('The run of record uses n_indep = 8,000; colab29 used 10,000.')
SWEEP_A


### Experiment A, plotted

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(SWEEP_A.n_indep, SWEEP_A.balanced, 'o-', color=COLOUR['Synth'])
ax[0].axvline(SYN_INDEP, ls='--', lw=1, color='0.5')
ax[0].text(SYN_INDEP, ax[0].get_ylim()[0], ' run of record', fontsize=8, color='0.5')
ax[0].set_xlabel('independent pairs generated'); ax[0].set_ylabel('balanced evaluation pairs')
ax[0].set_title('Does adding independent pairs grow the evaluation set?')

for i in range(4):
    ax[1].plot(SWEEP_A.n_indep, SWEEP_A[f'd{i}'], 'o-', label=f'decile {i}')
ax[1].axhline(STRAT_PER_BIN, ls='--', lw=1, color='0.5')
ax[1].text(0, STRAT_PER_BIN, ' cap = 400', fontsize=8, color='0.5', va='bottom')
ax[1].set_xlabel('independent pairs generated'); ax[1].set_ylabel('pairs available')
ax[1].set_title('Supply in the four lowest deciles')
ax[1].legend(frameon=False, fontsize=8)
for a in ax: a.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig('colab38_indep_sweep.png', dpi=150, bbox_inches='tight'); plt.show()


## 4. Experiment B - the decile cap

A cap only does something when a decile could supply more than the cap. Below the point where every
decile saturates, the cap is what equalises them; above it, the cap is inert and the evaluation set is
limited by whichever decile is scarcest.

In [ ]:
# EXPERIMENT B - where does the 400-per-decile cap start to bind?
NL_STD = np.concatenate([ALT, IND_MAX[:SYN_INDEP]])
OCC_STD = occupancy(NL_STD)
rows = []
for cap in [50, 100, 200, 400, 800, 1_600, 3_200, 10_000]:
    kept = np.minimum(OCC_STD, cap)
    rows.append(dict(cap=cap, balanced=int(kept.sum()),
                     deciles_saturated=int((OCC_STD >= cap).sum()),
                     smallest_decile=int(kept.min()),
                     ratio_largest_smallest=round(float(kept.max() / max(kept.min(), 1)), 2)))
SWEEP_B = pd.DataFrame(rows)
print('Available per decile in the standard set (20,000 altered + 8,000 independent):')
print('  ' + '  '.join(f'd{i}={OCC_STD[i]:,}' for i in range(10)))
print('')
print('"deciles_saturated" = deciles that could supply the full cap.')
print('"ratio_largest_smallest" = how UNEQUAL the balanced set is; 1.0 is perfect balance.')
SWEEP_B


## 5. Experiment C - the same question on CATH

Binned over the 200,000 sampled candidate pairs, which is the population the cap actually competes
with in the protocol.

In [ ]:
# EXPERIMENT C - the same question on the CATH collections.
# Protocol-faithful: bin the 200,000 sampled candidate pairs, which is what the
# cap actually competes with. NOT the exact all-pairs scan - that is colab37.
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')

def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq)
            and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))

COLLECTION = {
    'AA':  [s for d, s in zip(raw['domain_id'], raw['aa_seq'])              if _valid(s, is_aa, d)],
    'SS':  [s for d, s in zip(raw['domain_id'], raw['ss_seq'])              if _valid(s, is_ss, d)],
    '3Di': [s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)],
}
assert [len(COLLECTION[r]) for r in ['AA', 'SS', '3Di']] == [10_501, 10_497, 10_501], 'collections drifted'

def sampled_occupancy(seqs, n_cand=STRAT_CAND, seed=PAIR_SEED):
    rng = np.random.default_rng(seed); N = len(seqs)
    a = rng.integers(0, N, n_cand); b = rng.integers(0, N, n_cand)
    keep = a != b; a, b = a[keep], b[keep]
    nl = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    return occupancy(nl), nl

rows = {}
for r in ['AA', '3Di', 'SS']:
    print(f'binning {STRAT_CAND:,} candidate pairs for {r}...')
    occ, _ = sampled_occupancy(COLLECTION[r])
    rows[r] = occ
rows['Synth'] = OCC_STD

SUPPLY = pd.DataFrame(rows, index=[f'd{i}' for i in range(10)]).T
SUPPLY['deciles_at_cap'] = (SUPPLY[[f'd{i}' for i in range(10)]] >= STRAT_PER_BIN).sum(1)
SUPPLY['balanced'] = [balanced_size(SUPPLY.loc[r, [f'd{i}' for i in range(10)]].values, STRAT_PER_BIN)
                      for r in SUPPLY.index]
print('')
print('Pairs available per decile, before capping.')
print('⚠ The high deciles here are the SAMPLE only. The protocol also injects every known')
print('  high-similarity pair from the exact relevance sets before capping - the step that')
print('  rescues AA, which holds 5 such pairs in its entire collection (3Di 6,009, SS 623,077).')
SUPPLY


## 5b. Experiment B2 - the bound sweep on all four datasets

Experiment B swept the bound on Synth only, so Table B.2 reports one dataset and the sentence built
on it - *"nine of ten intervals are retained at the bound"* - is true of Synth and of no other.

**Experiment C above is not protocol-faithful and must not feed the sweep.** It bins the 200,000
sampled candidate pairs and stops there, but the protocol also injects every known high-similarity
pair before capping. Its own printed warning says so; its `balanced` column ignores it anyway. That
is why those numbers disagree with the evaluation sets actually used:

| | Experiment C `balanced` | run of record |
|---|---|---|
| AA | 1,211 | **1,216** |
| 3Di | 2,484 | **3,699** |
| SS | 3,381 | **4,000** |
| Synth | 3,648 | 3,648 |

Only Synth agrees, because Synth is generated pairwise and has nothing injected. For SS the injection
fills the top two intervals to the bound, and the count reaches the full 4,000. Sweeping the
un-injected supply would therefore report a bound sweep for a protocol the thesis does not use.

This cell takes the supply from `build_balanced` in colab40, which does the injection, and falls back
to recomputing it if the cache is gone.

**On what the sweep can show.** Both quantities usually cited are monotone in the bound - a larger
bound always retains more pairs and is always less even - so no bound optimises them jointly and
there is no interior optimum to find. What has structure is **which** intervals are retained at the
bound: a step function of the bound, constant over a range and then dropping. Each dataset has such a
range, and one protocol-wide bound has to sit in the intersection of the four.

In [ ]:
# EXPERIMENT B2, part 1 - the protocol-faithful supply, with the high-similarity pairs injected.
# Normal path: colab40 already computed it and cached it to Drive, so nothing is recomputed.
# Fallback needs the exact relevance-set scan (SS is 10,497 sequences all-against-all) and is
# therefore OPT-IN - set the flag below only if the cache is really gone and you have the time.
ALLOW_EXPENSIVE_REBUILD = False

import os, pickle, time
RANGE_HIGH = 0.70
DS    = ['AA', '3Di', 'SS', 'Synth']
DCOLS = [f'd{i}' for i in range(10)]

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/thesis_artefacts'
except Exception:
    CACHE = '/content/thesis_artefacts'
os.makedirs(CACHE, exist_ok=True)

STRAT_PATH, REL_PATH = f'{CACHE}/balanced_pairs.pkl', f'{CACHE}/relevance_sets.pkl'

if os.path.exists(STRAT_PATH):
    with open(STRAT_PATH, 'rb') as fh: STRAT = pickle.load(fh)
    SUPPLY_INJ = {r: np.array(STRAT[r]['supply'], dtype=int) for r in ['AA', 'SS', '3Di']}
    print(f'supply loaded from {STRAT_PATH} - nothing recomputed')
elif not ALLOW_EXPENSIVE_REBUILD:
    raise SystemExit(
        f'STOP. {STRAT_PATH} is not there, so the injected supply would have to be rebuilt.\n'
        f'  relevance_sets.pkl present: {os.path.exists(REL_PATH)}\n'
        '  With it, the rebuild is a few minutes. Without it, it is the full all-against-all\n'
        '  scan again (SS is 55M pairs).\n'
        '  Check the Drive folder first - colab40 wrote both files on 2026-08-25. If they are\n'
        '  really gone, set ALLOW_EXPENSIVE_REBUILD = True and rerun this cell.')
else:
    print('rebuilding the injected supply - this is the expensive path')
    from rapidfuzz.process import cdist as rf_cdist

    def build_relevance(seqs, block=1024, tag=''):
        # verbatim from colab40, pos_pairs only
        N = len(seqs); lens = np.array([len(s) for s in seqs]); pos = []
        t0 = time.time()
        for r0 in range(0, N, block):
            r1 = min(r0 + block, N)
            D = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
            den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
            sim = 1.0 - D / den
            for a in range(r1 - r0):
                i = r0 + a; row = sim[a].copy(); row[i] = -1.0
                for j in np.where(row >= RANGE_HIGH)[0]:
                    if j > i: pos.append((i, int(j), float(row[j])))
            print(f'    {tag} {r1:>6,}/{N:,}  ({time.time()-t0:.0f}s)', end='\r')
        print()
        return pos

    if os.path.exists(REL_PATH):
        with open(REL_PATH, 'rb') as fh: POS = {r: v['pos_pairs'] for r, v in pickle.load(fh).items()}
        print('relevance sets loaded from cache; only the candidate draw is recomputed')
    else:
        POS = {r: build_relevance(COLLECTION[r], tag=r) for r in ['AA', 'SS', '3Di']}
    for r, n in [('AA', 5), ('3Di', 6_009), ('SS', 623_077)]:
        assert len(POS[r]) == n, f'{r} high-similarity pair count differs from colab40 - STOP'

    # one shared rng consumed in colab40's order - AA, SS, 3Di - or the draw differs
    _rng = np.random.default_rng(PAIR_SEED)
    SUPPLY_INJ = {}
    for r in ['AA', 'SS', '3Di']:
        seqs = COLLECTION[r]; N = len(seqs)
        a = _rng.integers(0, N, STRAT_CAND); b = _rng.integers(0, N, STRAT_CAND)
        keep = a != b; a, b = a[keep], b[keep]
        nl = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
        if POS[r]:
            nl = np.concatenate([nl, np.array(POS[r], dtype=float)[:, 2]])
        bins = np.clip(np.digitize(nl, DECILES) - 1, 0, 9)
        SUPPLY_INJ[r] = np.bincount(bins, minlength=10)
        for bb in range(10):                      # colab40 draws the retained pairs here too
            idx = np.where(bins == bb)[0]
            if idx.size: _rng.permutation(idx)

SUPPLY_INJ['Synth'] = OCC_STD                     # generated pairwise, nothing to inject

OCC = {r: np.asarray(SUPPLY_INJ[r], dtype=int) for r in DS}
SUPPLY_FULL = pd.DataFrame({r: OCC[r] for r in DS}, index=DCOLS).T

RECORD = {'AA': 1_216, '3Di': 3_699, 'SS': 4_000, 'Synth': 3_648}   # colab40 evaluation_sizes
print('\nsupply per interval, high-similarity pairs included:')
print(SUPPLY_FULL.to_string())
print('')
ok = True
for r in DS:
    got = int(np.minimum(OCC[r], STRAT_PER_BIN).sum())
    ok &= got == RECORD[r]
    flag = 'ok' if got == RECORD[r] else f'MISMATCH, run of record is {RECORD[r]:,}'
    print(f'  {r:>5}: retained at bound {STRAT_PER_BIN} = {got:>6,}   {flag}')
print('')
print('all four reproduce the run of record' if ok else
      '⚠ STOP. This supply is not the one the reported results were built on. Reconcile before\n'
      '  any of it reaches the appendix.')


In [ ]:
# EXPERIMENT B2, part 2 - the sweep, and the range over which the bound behaves identically.
# Pure arithmetic on OCC: kept_k = min(supply_k, bound). No regeneration.
def cap_profile(occ, cap):
    kept = np.minimum(occ, cap)
    live = kept[occ > 0]                          # empty intervals are excluded from the ratio;
    return dict(retained=int(kept.sum()),         # AA has several, and they would make it infinite
                at_cap=int((occ >= cap).sum()),
                scarcest=int(live.min()),
                imbalance=round(float(kept.max() / max(live.min(), 1)), 1))

CAPS = [50, 100, 200, 400, 800, 1_600, 3_200, 10_000]
SWEEP_B2 = pd.DataFrame([
    {'cap': c, **{f'{r}_{k}': v for r in DS for k, v in cap_profile(OCC[r], c).items()}}
    for c in CAPS])
display(SWEEP_B2)

saturated = lambda occ, c: int((occ >= c).sum())
BASE = {r: saturated(OCC[r], STRAT_PER_BIN) for r in DS}

def flat_window(pred, start=STRAT_PER_BIN, hard=200_000):
    lo = hi = start
    while lo > 1    and pred(lo - 1): lo -= 1
    while hi < hard and pred(hi + 1): hi += 1
    return lo, hi

WINDOW = {r: flat_window(lambda c, r=r: saturated(OCC[r], c) == BASE[r]) for r in DS}
JOINT  = flat_window(lambda c: all(saturated(OCC[r], c) == BASE[r] for r in DS))

print(f'intervals retained at the bound, at {STRAT_PER_BIN}: '
      + ', '.join(f'{r} {BASE[r]}' for r in DS))
print('')
for r in DS:
    lo, hi = WINDOW[r]; live = OCC[r][OCC[r] > 0]
    print(f'  {r:>5}: unchanged over bounds [{lo:,}, {hi:,}]   '
          f'non-empty intervals {len(live)}   scarcest {live.min():,}')
print('')
print(f'  INTERSECTION: [{JOINT[0]:,}, {JOINT[1]:,}]')
print('')
print('Retained size is increasing in the bound, so within the intersection the largest value')
print('is the one keeping every dataset at its saturation profile while retaining the most pairs:')
for c in [JOINT[0], STRAT_PER_BIN, JOINT[1], JOINT[1] + 1]:
    mark = '   <- the value used' if c == STRAT_PER_BIN else ''
    print(f'    bound {c:>6,}: ' + '  '.join(
        f'{r} {cap_profile(OCC[r], c)["retained"]:>6,} ({cap_profile(OCC[r], c)["at_cap"]})'
        for r in DS) + mark)
print('')
print('⚠ This is a PROPERTY of the value, not its origin. Nothing on record says it was chosen')
print('  this way, and the appendix must not claim it was.')
print('')
print('=== appendix table rows ===')
for c in CAPS:
    cells = [f"{cap_profile(OCC[r], c)['retained']:,} & {cap_profile(OCC[r], c)['at_cap']}"
             for r in DS]
    print(f"{c:,} & " + ' & '.join(cells) + r' \\')


## 5c. Experiment E - the candidate count, and the population behind it

Two questions, and they are about different halves of the range.

**The population.** The bound and the candidate count both act on a supply, and the supply is a sample
from a fixed population: every pair of sequences in the collection, about 55 million per dataset. For
the intervals at or above 0.70 that population is **known exactly** - the relevance scan enumerated
it, which is how the injection is possible at all. For the intervals below 0.70 it is estimated by
scaling the sample, and the scale factor is large enough (roughly 275x at 200,000 candidates) that
the estimate is precise wherever the count is not tiny. The table below reports both, marked.

**The candidate count.** Raising it can only ever *add* pairs to an interval, so it can never reduce
the number of intervals retained at the bound. The only thing it can do is push a currently short
interval up to the bound. Above 0.70 it cannot even do that, because the injection already supplies
every pair there. So the whole question reduces to: **is there an interval below 0.70 that 200,000
candidates leave short and a larger draw would fill?**

The sweep is nested - one draw of `N_MAX` per dataset, prefixes taken - so differences between rows
are the effect of the count and not of a different draw.

⚠ Synth is not in this sweep. It is generated pairwise rather than sampled from a collection, and its
count is Experiment A's `n_indep`.

In [ ]:
# EXPERIMENT E, part 1 - the candidate-count sweep. One nested draw per collection.
# Cost is N_MAX normLev per dataset. 1,000,000 takes a few minutes each; lower N_MAX if impatient.
N_MAX = 1_000_000
N_GRID = [100_000, 200_000, 400_000, 700_000, 1_000_000]
CATH = ['AA', '3Di', 'SS']

# the injected pairs, binned - needed to separate what sampling can reach from what it cannot
if os.path.exists(REL_PATH):
    with open(REL_PATH, 'rb') as fh: _rel = pickle.load(fh)
    INJ_HIST = {r: occupancy(np.array([p[2] for p in _rel[r]['pos_pairs']]))
                if _rel[r]['pos_pairs'] else np.zeros(10, dtype=int) for r in CATH}
    del _rel
    print('injected pairs, binned:')
    print(pd.DataFrame(INJ_HIST, index=DCOLS).T.to_string())
else:
    INJ_HIST = {r: np.zeros(10, dtype=int) for r in CATH}
    print('⚠ relevance_sets.pkl absent - the sweep below is SAMPLING ONLY, without the injection.')
    print('  Read it as "what a random draw reaches", not as the protocol supply.')

NL_BIG, rows = {}, []
for r in CATH:
    rng = np.random.default_rng(PAIR_SEED); seqs = COLLECTION[r]; N = len(seqs)
    a = rng.integers(0, N, N_MAX); b = rng.integers(0, N, N_MAX)
    keep = a != b; a, b = a[keep], b[keep]
    t0 = time.time()
    NL_BIG[r] = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    print(f'  {r}: {len(NL_BIG[r]):,} candidate pairs scored ({time.time()-t0:.0f}s)')

for r in CATH:
    for n in N_GRID:
        occ = occupancy(NL_BIG[r][:n])
        tot = occ + INJ_HIST[r]
        rows.append(dict(dataset=r, n_cand=n,
                         **{f'd{i}': int(tot[i]) for i in range(10)},
                         at_bound=int((tot >= STRAT_PER_BIN).sum()),
                         short_below_070=int((tot[:7] < STRAT_PER_BIN).sum()),
                         retained=int(np.minimum(tot, STRAT_PER_BIN).sum())))
SWEEP_E = pd.DataFrame(rows)
display(SWEEP_E)

print('short_below_070 = intervals under 0.70 that cannot supply the bound. These are the only')
print('ones a larger draw could rescue; everything at or above 0.70 is already complete.')
print('')
for r in CATH:
    sub = SWEEP_E[SWEEP_E.dataset == r]
    lo, hi = sub.iloc[0], sub.iloc[-1]
    verdict = ('no change' if lo.at_bound == hi.at_bound else
               f'{lo.at_bound} -> {hi.at_bound} intervals at the bound')
    print(f'  {r:>5}: {N_GRID[0]:,} to {N_GRID[-1]:,} candidates, a {N_GRID[-1]//N_GRID[0]}-fold '
          f'increase: {verdict}; retained {lo.retained:,} -> {hi.retained:,}')


In [ ]:
# EXPERIMENT E, part 2 - the population behind the sample.
# Below 0.70: estimated from the 1,000,000-pair draw, scaled to all pairs, with a binomial
# 95% interval so the precision is visible rather than implied.
# At or above 0.70: the exact enumerated count, not an estimate.
from math import comb

POP = {}
for r in CATH:
    n = len(NL_BIG[r]); total = comb(len(COLLECTION[r]), 2)
    occ = occupancy(NL_BIG[r]); p = occ / n
    se = np.sqrt(np.maximum(p * (1 - p), 0) / n)
    est, lo, hi = p * total, (p - 1.96 * se) * total, (p + 1.96 * se) * total
    row = {}
    for i in range(10):
        if i >= 7 and INJ_HIST[r].sum():                    # exact, from the relevance scan
            row[f'd{i}'] = f'{INJ_HIST[r][i]:,}'
        else:
            row[f'd{i}'] = (f'{est[i]:,.0f}' if occ[i] >= 30 else
                            f'~{est[i]:,.0f} [{max(lo[i],0):,.0f}-{hi[i]:,.0f}]' if occ[i]
                            else '0 in sample')
    row['all pairs'] = f'{total:,}'
    POP[r] = row

print('Estimated population per interval, whole collection.')
print('Intervals 7-9 are EXACT (enumerated). Intervals 0-6 are scaled from the '
      f'{len(NL_BIG[CATH[0]]):,}-pair draw;')
print('a 95% interval is shown where the sample count is under 30, and those are the only')
print('cells where a larger draw would change the picture.')
display(pd.DataFrame(POP).T)

print('')
print('⚠ The estimate assumes the draw is uniform over pairs, which it is. It is NOT an exact')
print('  all-pairs histogram below 0.70 - producing one means scanning ~55M pairs per dataset.')
print('  Say "estimated" in the caption or do the scan; do not print it as exact.')


In [ ]:
# EXPERIMENT E, part 3 - plotted. Locked palette (design file section 1).
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

for r in CATH:                                    # left: does a bigger draw buy intervals?
    sub = SWEEP_E[SWEEP_E.dataset == r]
    ax[0].plot(sub.n_cand, sub.at_bound, 'o-', color=COLOUR[r], label=r)
ax[0].axvline(STRAT_CAND, ls='--', lw=1, color='0.5')
ax[0].text(STRAT_CAND, ax[0].get_ylim()[0], ' protocol', fontsize=8, color='0.5')
ax[0].set_ylim(0, 10.5); ax[0].set_yticks(range(0, 11, 2))
ax[0].set_xlabel('candidate pairs drawn'); ax[0].set_ylabel('intervals at the bound')
ax[0].set_title('Does a larger draw balance more intervals?')
ax[0].legend(frameon=False, fontsize=8)

for r in CATH:                                    # middle: and does it buy pairs?
    sub = SWEEP_E[SWEEP_E.dataset == r]
    ax[1].plot(sub.n_cand, sub.retained, 'o-', color=COLOUR[r], label=r)
ax[1].axhline(10 * STRAT_PER_BIN, ls=':', lw=1, color='0.5')
ax[1].text(N_GRID[0], 10 * STRAT_PER_BIN, ' ceiling, 10 x bound', fontsize=8, color='0.5',
           va='bottom')
ax[1].axvline(STRAT_CAND, ls='--', lw=1, color='0.5')
ax[1].set_xlabel('candidate pairs drawn'); ax[1].set_ylabel('evaluation pairs retained')
ax[1].set_title('Does a larger draw grow the evaluation set?')

# right: where each dataset actually sits, at the protocol count
w = 0.27
for k, r in enumerate(CATH):
    sub = SWEEP_E[(SWEEP_E.dataset == r) & (SWEEP_E.n_cand == STRAT_CAND)].iloc[0]
    ax[2].bar(np.arange(10) + (k - 1) * w, [min(sub[f'd{i}'], STRAT_PER_BIN) for i in range(10)],
              width=w, color=COLOUR[r], label=r)
ax[2].axhline(STRAT_PER_BIN, ls='--', lw=1, color='0.5')
ax[2].set_xticks(range(10)); ax[2].set_xlabel('similarity interval')
ax[2].set_ylabel('pairs retained'); ax[2].set_title(f'Retained per interval at {STRAT_CAND:,}')
ax[2].legend(frameon=False, fontsize=8)

for a in ax:
    a.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig('colab38_cand_sweep.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# EXPERIMENT F - the EXACT population per interval, every pair of every collection.
# colab37 already ran this scan (exact_hist, SAMPLE_ONLY=False) and saved only its metadata,
# so the histogram was lost. colab40's relevance scan computes the same similarities and keeps
# only those >= 0.70. This cell runs it once more and PERSISTS the counts, so it is never
# recomputed again. Cost: one full scan per collection, SS is the slow one (tens of minutes).
RUN_EXACT_POPULATION = False        # set True to spend the time
POP_PATH = f'{CACHE}/population_hist.pkl'
BINS_FINE = 2_000                   # fine grid alongside the deciles, so a future question
                                    # about any threshold is answerable without rescanning

if os.path.exists(POP_PATH):
    with open(POP_PATH, 'rb') as fh: POPHIST = pickle.load(fh)
    print(f'exact population loaded from {POP_PATH} - nothing recomputed')
elif not RUN_EXACT_POPULATION:
    POPHIST = None
    print('skipped. population_estimate above stays an ESTIMATE below 0.70.')
    print('Set RUN_EXACT_POPULATION = True to replace it with exact counts.')
else:
    from rapidfuzz.process import cdist as rf_cdist

    def exact_population(seqs, block=1024, tag=''):
        """Every unordered pair, binned two ways. Decile bins use decile_of, NOT
           np.histogram - the two disagree at exactly 0.70 (the float edge), and the
           protocol uses decile_of."""
        lens = np.array([len(s) for s in seqs]); N = len(seqs)
        dec  = np.zeros(10, dtype=np.int64)
        fine = np.zeros(BINS_FINE, dtype=np.int64)
        fine_edges = np.linspace(0, 1, BINS_FINE + 1)
        n_high = 0; total = 0; t0 = time.time()
        for r0 in range(0, N, block):
            r1 = min(r0 + block, N)
            D = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
            den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
            sim = 1.0 - D / den
            keep = np.arange(N)[None, :] > np.arange(r0, r1)[:, None]   # each pair once
            v = sim[keep]
            dec    += np.bincount(decile_of(v), minlength=10)
            fine   += np.histogram(v, bins=fine_edges)[0]
            n_high += int((v >= RANGE_HIGH).sum())
            total  += int(v.size)
            print(f'    {tag} rows {r1:>6,}/{N:,}  ({time.time()-t0:.0f}s)', end='\r')
        print()
        return dict(decile=dec, fine=fine, n_pairs=total, n_high=n_high)

    POPHIST = {}
    for r in CATH:
        print(f'{r}: exact scan over {len(COLLECTION[r]):,} sequences')
        POPHIST[r] = exact_population(COLLECTION[r], tag=r)
        assert POPHIST[r]['n_pairs'] == comb(len(COLLECTION[r]), 2), f'{r} pair count wrong'
        assert POPHIST[r]['n_high'] == {'AA': 5, '3Di': 6_009, 'SS': 623_077}[r], \
            f'{r} high-range count differs from the relevance scan - STOP'
    with open(POP_PATH, 'wb') as fh: pickle.dump(POPHIST, fh)
    print(f'cached to {POP_PATH} - the artefact colab37 failed to keep')

if POPHIST is not None:
    POP_EXACT = pd.DataFrame({r: POPHIST[r]['decile'] for r in CATH}, index=DCOLS).T
    POP_EXACT['all pairs'] = [POPHIST[r]['n_pairs'] for r in CATH]
    print('\nExact population per interval, every pair of every collection:')
    print(POP_EXACT.to_string())
    print('')
    # intervals 7-9 do NOT equal the >=0.70 count: exactly-0.70 pairs bin into interval 6.
    # The gap is that set, and reporting it measures the float edge on the whole population.
    for r in CATH:
        d = POPHIST[r]['decile']
        print(f'  {r:>5}: pairs >= 0.70 = {POPHIST[r]["n_high"]:>9,}   intervals 7-9 = '
              f'{int(d[7:].sum()):>9,}   exactly 0.70, binned into interval 6 = '
              f'{POPHIST[r]["n_high"] - int(d[7:].sum()):>7,}')
    print('')
    print('AA, the two readings of "mid range":')
    d = POPHIST['AA']['decile']
    print(f'  Table 3.1 mid range [0.30, 0.70) = intervals 3-6 = {int(d[3:7].sum()):,}')
    print(f'  intervals 4-6 only,   [0.40, 0.70) = {int(d[4:7].sum()):,}')
    print('  Whichever of these is ~50 is the one the appendix may call by that number.')


## 6. Experiment D - the match to the training set

The hypothesis: 20,000 + 10,000 = 30,000 was chosen to match `N_TRAIN`. If that was the intent, the
best `n_indep` is the one whose decile profile sits closest to the training set's. Total variation
distance over the ten deciles measures it. A flat or minimum-elsewhere result means the arithmetic
coincidence was not a design criterion - which is a perfectly good thing to report, as long as it is
reported rather than rationalised after the fact.

In [ ]:
# EXPERIMENT D - was 20,000 + 10,000 = 30,000 sized to match the training set?
# If the intent was to reproduce the training distribution, the best n_indep is the
# one whose decile profile is closest to it. Measured, not assumed.
train_p = occupancy(TRAIN_NL) / len(TRAIN_NL)
rows = []
for n_ind in [0, 2_000, 4_000, 6_000, 8_000, 10_000, 15_000, 20_000]:
    nl = np.concatenate([ALT, IND_MAX[:n_ind]])
    p = occupancy(nl) / len(nl)
    tv = 0.5 * np.abs(p - train_p).sum()          # total variation distance over deciles
    rows.append(dict(n_indep=n_ind, total=len(nl),
                     median=round(float(np.median(nl)), 4),
                     tv_distance_to_training=round(float(tv), 4)))
MATCH = pd.DataFrame(rows)
best = MATCH.loc[MATCH.tv_distance_to_training.idxmin(), 'n_indep']
print(f'training-set median {np.median(TRAIN_NL):.4f}')
print(f'closest match to the training decile profile: n_indep = {best:,}')
print('')
print('If the answer is not 8,000 or 10,000, then the totals matching N_TRAIN = 30,000')
print('was a coincidence of arithmetic rather than a measured design choice. Say so plainly.')
MATCH


## 7. Save, and the sentences this licenses

In [ ]:
summary = dict(
    indep_sweep=SWEEP_A.to_dict('records'),
    cap_sweep=SWEEP_B.to_dict('records'),                       # Synth, sampled candidates only
    cap_sweep_all=SWEEP_B2.to_dict('records'),                  # all four, injection included
    cand_sweep=SWEEP_E.to_dict('records'),                      # candidate count, CATH only
    population_estimate={r: POP[r] for r in CATH},              # scaled from the 1M draw
    population_exact=(None if POPHIST is None else
                      {r: dict(decile=POPHIST[r]['decile'].tolist(),
                               n_pairs=POPHIST[r]['n_pairs'],
                               n_high=POPHIST[r]['n_high']) for r in CATH}),
    injected_per_interval={r: INJ_HIST[r].tolist() for r in CATH},
    supply_per_decile=SUPPLY.reset_index().rename(columns={'index': 'dataset'}).to_dict('records'),
    supply_per_decile_injected=SUPPLY_FULL.reset_index()
        .rename(columns={'index': 'dataset'}).to_dict('records'),
    saturation_windows={r: list(WINDOW[r]) for r in DS},
    saturation_window_joint=list(JOINT),
    training_match=MATCH.to_dict('records'),
)
with open('colab38_protocol_constants.json', 'w') as fh:
    json.dump(summary, fh, indent=2, default=int)

print('=== sentences the appendix can now make ===')
a8  = SWEEP_A[SWEEP_A.n_indep == SYN_INDEP].iloc[0]
a10 = SWEEP_A[SWEEP_A.n_indep == 10_000].iloc[0]
print(f'  * 8,000 independent pairs yield {a8.balanced:,} balanced pairs; '
      f'10,000 yield {a10.balanced:,}.')
print(f'  * intervals retained at the bound: ' + ', '.join(f'{r} {BASE[r]}' for r in DS)
      + ' - the Synth figure is not general and Table B.2 must stop implying it is.')
print(f'  * every dataset keeps that count over bounds [{JOINT[0]:,}, {JOINT[1]:,}]; '
      f'{STRAT_PER_BIN} is the largest round value inside it.')
print(f'  * the residual imbalance is set by the scarcest interval of each collection ('
      + ', '.join(f'{r} {int(OCC[r][OCC[r] > 0].min()):,}' for r in DS)
      + '), which no bound can change.')
for r in CATH:
    sub = SWEEP_E[SWEEP_E.dataset == r]; lo, hi = sub.iloc[0], sub.iloc[-1]
    print(f'  * {r}: a {N_GRID[-1]//N_GRID[0]}-fold larger candidate draw moves the intervals at '
          f'the bound {lo.at_bound} -> {hi.at_bound} and the set {lo.retained:,} -> {hi.retained:,}.')
aa = SWEEP_E[SWEEP_E.dataset == 'AA']
print(f'  * AA interval 0 supplies {int(aa.iloc[0].d0)} pairs at {N_GRID[0]:,} candidates and '
      f'{int(aa.iloc[1].d0)} at {N_GRID[1]:,}; it crosses the bound at about '
      f'{np.interp(STRAT_PER_BIN, aa.d0.values, aa.n_cand.values):,.0f}.')
print(f'  * the lowest interval is limited by the alphabet, not by sampling: '
      f'{int((IND_MAX < 0.10).sum())} of 20,000 independent pairs fall below 0.10.')
print('')
print('⚠ supply_per_decile (Table B.1) omits the injection and disagrees with the run of record.')
print('  supply_per_decile_injected is the one the appendix should print.')
if POPHIST is None:
    print('⚠ population_estimate is EXACT at intervals 7-9 and ESTIMATED below. Caption it that')
    print('  way, or rerun with RUN_EXACT_POPULATION = True to replace it.')
else:
    print('✓ population_exact is present - the appendix can print exact counts throughout.')
print('⚠ SWEEP_E redraws candidates at each size, so its 200,000 row will not match Table B.1')
print('  exactly. Caption it as an independent draw per size.')
print('')
print('Download colab38_protocol_constants.json + colab38_indep_sweep.png + colab38_cand_sweep.png')
